# 02 - Open boundary conditions: sanity checks and comparison with the interior

Check the MOM6 OBC segment files for extreme values and fill artifacts, then ask whether the interior of the run actually follows them.

The segment files are awkward to read -- every variable and dimension is named after its segment -- but the geometry is simple: a boundary runs along the edge of the model grid, on supergrid points. Taking every other point puts it straight onto the model's tracer cells, so the whole notebook works without interpolating anything in space.


## Setup

History files are opened directly in the cells below; only the scientific operations are wrapped in functions.

In [ ]:
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}
import glob
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean.cm as cmo
import matplotlib.ticker as mticker
import xgcm

from mom6_tools.wright_eos import wright_eos, alpha_wright_eos, beta_wright_eos
from mom6_tools.jobqueue import get_cluster

xr.set_options(keep_attrs=True)

FIGDIR = Path("diagnostic_figures")
FIGDIR.mkdir(exist_ok=True)

### Start a Dask Cluster/Client
Dask is a parallel computation library for Python that lets us initialize multiple workers to act as parallel processors. There are two options for creating a dask cluster:
- PBSCluster: This submits new PBS jobs for each worker, running separate jobs on Casper/Derecho for each worker.
- LocalCluster: If you are running this notebook on an existing PBS job with enough resources, it can access those resources directly.

In [ ]:
DASK = "PBSCluster" # or LocalCluster (works with SLURMCluster on other systems)

N_WORKERS = 4

ACCOUNT = "P93300012" # charge worker jobs here, defaults to PROJECT and PBS_ACCOUNT env variables in bash
QUEUE = "casper" # or "derecho"
INTERFACE = 'ext' # or "ib0" for derecho

# Following are resources for each worker
# Total resources are N_WORKERS * resource (e.g. N_WORKERS * MEMORY).
MEMORY = "20GB"
CORES = 4
PROCESSES = 1
WALLTIME = "03:00:00"

success, client, cluster = get_cluster(nw=N_WORKERS, cluster_class=DASK, account=ACCOUNT, queue=QUEUE, walltime=WALLTIME, memory=MEMORY, processes=PROCESSES, cores = CORES, interface=INTERFACE, log_directory="logs",)

In [ ]:
client

## Case Selection

In [ ]:
# --- Case configuration ---
CASE = dict(
        casename="small_alaska",
        hist="/glade/derecho/scratch/manishrv/archive/small_alaska/ocn/hist",
        input_dir="/glade/u/home/manishrv/scratch/croc_input/small_alaska/ocn",
        date_range=["2020-01-01", "2020-01-08"],
        length="demo" # or "full", if you have more than 1 month of output. Otherwise native and z files will be filled values
    )

HIST = CASE["hist"]
INPUT_DIR = CASE["input_dir"]
DATE_RANGE = CASE["date_range"]
FILE_PATTERN = f"{HIST}/{CASE['casename']}.mom6"
MONTH = DATE_RANGE[0][:7]   # the monthly native/z file DATE_RANGE starts in

## Other Paths and Settings
GLORYS_ROOT = "/gdex/data/d010049"

# MOM6/FMS counts days with proleptic-Gregorian rules but labels the axis
# `calendar = "gregorian"`, which CF defines as the mixed Julian/Gregorian
# calendar. Open history files with decode_times=False and pass them to decode_mom6_time().
# If you are running with 'noleap', use `MOM6_CALENDAR = "noleap"`
MOM6_CALENDAR = "proleptic_gregorian"

print(f"{CASE['casename']}: {FILE_PATTERN}.h.*.nc")

### Relevant Files
- STATIC - `casename.mom6.h.static.nc`: contains real lat/lon coords, grid information, coriolis parameter
- GEOM - `casename.mom6.h.ocean_geometry.nc`: a lot of similar info to STATIC, includes bathymetry

The provided geolat/geolon coords give physical lat/lon adjusted across the grid for the most accurate plotting and geolocating. Default xh/yh coords are the nominal lat/lon which are best used as indices and temporary coords before rigorous scientific plotting/analysis.

In [ ]:
# --- Grid files ---
# The static file carries the land masks, cell areas and Coriolis parameter.
STATIC = xr.open_dataset(f"{FILE_PATTERN}.h.static.nc", decode_times=False).squeeze(drop=True)

# Rename geometry coords to the history-file names.
GEOM = (xr.open_dataset(f"{FILE_PATTERN}.h.ocean_geometry.nc")
          .rename_dims({"lath": "yh", "lonh": "xh", "latq": "yq", "lonq": "xq"})
          .drop_vars(["lath", "lonh", "latq", "lonq"]))

# Attach these to any history DATASET with `ds = ds.assign_coords(COORDS)`
# This streamlines plotting and xgcm grid operations, which expect the geometry coords to be present.
COORDS = {name: GEOM[name] for name in
          ("geolon", "geolat", "geolonu", "geolatu",
           "geolonv", "geolatv", "geolonb", "geolatb")}

# Example: STATIC now stores geolon/geolat as data variables (psst it already had them)
STATIC = STATIC.assign_coords(COORDS)

print(dict(STATIC.sizes))
print("lon {:.2f} .. {:.2f}   lat {:.2f} .. {:.2f}".format(
    float(GEOM.geolon.min()), float(GEOM.geolon.max()),
    float(GEOM.geolat.min()), float(GEOM.geolat.max())))

### Formatting Plots
For efficiency in these notebooks, we provide some standard plot settings and information. This lets us quickly use the implicit plotting capapbilities of xarray (i.e. dataarray.plot(kwargs**)).

In [ ]:
# --- Standard plotting keywords ---
# Everything is plotted with xarray's own `.plot()`; these dicts just supply the
# coordinates and the transform so everything is standardized and pretty!

SUBPLOT = {"projection": ccrs.Robinson(central_longitude=float(GEOM.geolon.mean()))}
LAND = cfeature.LAND.with_scale("50m")

_aspect = float(GEOM.geolat.max() - GEOM.geolat.min()) / float(GEOM.geolon.max() - GEOM.geolon.min())
PANEL = np.array([18.0, round(18.0 * _aspect)])

# Lat lon gridlines for all plots. Use `ax.gridlines(**GRID)` to apply.
nticks = 7
GRID = dict(
      draw_labels=["bottom", "left"],
      xlocs=mticker.MultipleLocator(np.round(np.abs((GEOM.geolon.max().values - GEOM.geolon.min().values) / nticks), 1)),
      ylocs=mticker.MultipleLocator(np.round(np.abs((GEOM.geolat.max().values - GEOM.geolat.min().values) / nticks), 1)),
      linewidth=0.5, color="0.5", alpha=0.5,
  )

# Fast map for plotting MOM6 fields. Use `ds.plot(**MAP[grid])` where grid is one of "h", "u", "v", or "q".
# h - tracers, q - corners, u - u-velocity, v - v-velocity
MAP = {
    "h": dict(x="geolon",  y="geolat",  transform=ccrs.PlateCarree()),
    "u": dict(x="geolonu", y="geolatu", transform=ccrs.PlateCarree()),
    "v": dict(x="geolonv", y="geolatv", transform=ccrs.PlateCarree()),
    "q": dict(x="geolonb", y="geolatb", transform=ccrs.PlateCarree()),
}

# add-ons to spread over a `.plot()` call
DIFF = dict(cmap="cmo.balance", center=0, robust=True)   # diverging / difference
SECTION = dict(yincrease=False)                          # depth down the y axis

### xgcm grid and metrics
We use the xgcm for efficient handling of the Arakawa C-grid geometry. We provide information about which corrdinates correspond to tracer (center) points velocity (face) points. We also provide grid metrics from the geometry file into the xgcm grid and the relevant datasets.
Metrics give grid cell spacing and areas for different operations (e.g. area-weighted interpolation and grid size based difference).

The coordinates and metrics in the xgcm grid need to be present in the history file.

In [ ]:
# --- xgcm grid and derived quantities ---

XGCM_COORDS = {
    "X": {"center": "xh", "outer": "xq"},
    "Y": {"center": "yh", "outer": "yq"},
    "Z": {"center": "zl", "outer": "zi"},
}

def make_grid(ds, geom, with_metrics=True, xgcm_coords=XGCM_COORDS):
    """Build an xgcm.Grid for a MOM6 regional dataset, as (grid, ds_with_metrics).

    Metrics come from the ocean_geometry file `geom` so that grid.diff /
    grid.interp / grid.derivative / grid.integrate know the cell spacings.
    """
    coords = {ax: pos for ax, pos in xgcm_coords.items() if pos["center"] in ds.dims} # flexible for 3D/2D data
    obj, metrics = ds, None
    
    if with_metrics:
        for v in ("dxCu", "dyCu", "dxCv", "dyCv", "dxT", "dyT",
                  "dxBu", "dyBu", "Ah", "Aq"):
            if v in geom and set(geom[v].dims) <= set(ds.dims):
                obj = obj.assign_coords({v: geom[v]})
        metrics = {
            ("X",): [v for v in ("dxT", "dxCu", "dxCv", "dxBu") if v in obj.coords],
            ("Y",): [v for v in ("dyT", "dyCu", "dyCv", "dyBu") if v in obj.coords],
            ("X", "Y"): [v for v in ("Ah", "Aq") if v in obj.coords],
        }
        metrics = {k: v for k, v in metrics.items() if v}
        
    grid = xgcm.Grid(obj, coords=coords, metrics=metrics, padding="extend",
                autoparse_metadata=False)
    
    return grid, obj

def decode_mom6_time(ds, calendar="proleptic_gregorian"):
    """Decode MOM6 time axes, correcting the calendar that MOM6 mislabels.
    """
    ds = ds.copy()
    for v in ds.variables:
        attrs = dict(ds[v].attrs)
        if "since" in str(attrs.get("units", "")):
            attrs["calendar"] = calendar
            ds[v].attrs = attrs
    return xr.decode_cf(ds)

In [ ]:
# --- Reading an OBC segment file ---
# Everything in forcing_obc_segment_*.nc is named after its segment
# (`temp_segment_001`, `nz_segment_001_temp`, `lon_segment_001`, ...), and the
# boundary runs along whichever of `nx_segment_*` / `ny_segment_*` is longer than
# one point.  `open_obc` strips all of that down to dims (time, depth, n).
OBC_VARS = ("temp", "salt", "u", "v", "eta")


def open_obc(path):
    """Read one OBC segment file as a Dataset with dims (time, depth, n).

    `n` counts points along the boundary; `lon`, `lat` and `depth` come along as
    coordinates.  The attributes record the segment id and its orientation:
    "zonal" = constant latitude, "meridional" = constant longitude, with
    `const_coord` holding that constant value.
    """
    sid = re.search(r"segment_(\d+)\.nc$", path).group(1)
    raw = xr.open_dataset(path).squeeze(drop=True)   # drops the 1-point cross-boundary dim
    along = (f"nx_segment_{sid}" if raw.sizes.get(f"nx_segment_{sid}", 1) > 1
             else f"ny_segment_{sid}")

    def tidy(name):
        """One field, with its segment-specific dim names replaced by n / depth."""
        da = raw[f"{name}_segment_{sid}"]
        return da.rename({along: "n",
                          **{d: "depth" for d in da.dims if d.startswith("nz")}})

    ds = xr.Dataset({v: tidy(v) for v in OBC_VARS}).assign_coords(
        n=np.arange(raw.sizes[along]),
        depth=raw["depth"].values,
        lon=("n", raw[f"lon_segment_{sid}"].values.ravel()),
        lat=("n", raw[f"lat_segment_{sid}"].values.ravel()),
    )

    # The files label their time axis "julian" or "gregorian"; MOM6 reads the
    # dates at face value, so convert date-for-date onto the history calendar.
    ds = ds.convert_calendar("proleptic_gregorian", use_cftime=False)

    zonal = float(np.ptp(ds["lat"].values)) < 1e-6
    ds.attrs.update(segment=sid,
                    orientation="zonal" if zonal else "meridional",
                    const_coord=float(ds["lat"][0] if zonal else ds["lon"][0]))
    return ds


def obc_label(ds):
    """Human-readable label for a segment dataset."""
    key = "lat" if ds.attrs["orientation"] == "zonal" else "lon"
    return f"segment {ds.attrs['segment']} ({key} = {ds.attrs['const_coord']:g})"


def along_name(ds):
    """Name of the coordinate that varies along the segment."""
    return "lon" if ds.attrs["orientation"] == "zonal" else "lat"


def normal_var(ds):
    """The velocity component normal to the segment."""
    return "v" if ds.attrs["orientation"] == "zonal" else "u"


In [ ]:
# from mom6_tools.regional_m6toolbox import obc_label

# --- Putting a segment on the model's tracer cells ---
# The boundary points are *supergrid* points, at twice the resolution of the
# model grid: a segment of 2N+1 points spans N tracer cells, and points
# 1, 3, 5, ... sit exactly on the cell centres.  So every other point lines a
# segment up one-for-one with a row (or column) of model h-points, and nothing
# has to be interpolated in space anywhere below.

# The topography the OBC files were generated against, renamed onto the
# history-file dimension names -- the t-cells of the two files are the same cells.
TOPOG = xr.open_dataset(sorted(glob.glob(f"{INPUT_DIR}/ocean_topog_*.nc"))[0]
                        ).rename({"ny": "yh", "nx": "xh"})


def to_t_points(seg):
    """Every other boundary point: the segment on the model's tracer cells."""
    out = seg.isel(n=slice(1, None, 2))
    return out.assign_coords(n=np.arange(out.sizes["n"]))


def on_boundary(obj, seg, topog):
    """Slice a t-point field (`topog` itself, or a history stream) along a segment.

    Returns the row (or column) of t-cells that lies along the boundary,
    relabelled with the segment's own `n`, `lon` and `lat` so that it can be
    differenced against the segment directly.  (Those are the boundary-edge
    positions; the cell centres sit half a cell inside.)

    `topog` is the ocean_topog dataset the OBC files were generated against,
    renamed onto the history-file dimension names; its `x`/`y` are what the
    boundary index is looked up in.
    """
    if seg.attrs["orientation"] == "zonal":       # constant latitude -> one yh row
        dim = "yh"
        k = int(np.abs(topog["y"].isel(xh=0) - seg.attrs["const_coord"]).argmin("yh"))
    else:                                         # constant longitude -> one xh column
        dim = "xh"
        k = int(np.abs(topog["x"].isel(yh=0) - seg.attrs["const_coord"]).argmin("xh"))
    along = "xh" if dim == "yh" else "yh"

    out = obj.isel({dim: k}).rename({along: "n"})
    if out.sizes["n"] != seg.sizes["n"]:
        raise ValueError(
            f"{obc_label(seg)} has {seg.sizes['n']} tracer cells but the boundary "
            f"row has {out.sizes['n']} -- does the segment span the whole edge?")
    out.attrs["boundary_line"] = f"{dim} index {k}"
    return out.assign_coords(n=seg["n"].values,
                             lon=("n", seg["lon"].values),
                             lat=("n", seg["lat"].values))


def mask_below_floor(seg, floor):
    """Blank the levels of a segment that lie below the sea floor, and land points.

    The OBC files carry no bathymetry: every level at every point holds a value,
    so on a shallow shelf most of a 50-level GLORYS-derived file is extrapolated
    fill -- small_alaska is 121 m deep but its boundary files run to 5728 m.
    """
    wet = floor["mask"] > 0
    out = seg[["temp", "salt", "u", "v"]].where(wet & (seg["depth"] < floor["depth"]))
    out["eta"] = seg["eta"].where(wet)             # eta is 2-D, so only the land mask
    out.attrs = dict(seg.attrs)
    return out


## 1. Segments on disk

Read every `forcing_obc_segment_*.nc` file, drop it onto the model's tracer cells, keep a short slice of the time axis, and summarise what each segment contains.


In [ ]:
# from mom6_tools.regional_m6toolbox import open_obc, to_t_points, obc_label

# Number of OBC time records to load.  The files hold the whole run at daily
# resolution, so this is the main cost knob in this notebook.
NTIME = 30

SEG_RAW = {}
for path in sorted(glob.glob(f"{INPUT_DIR}/forcing_obc_segment_*.nc")):
    seg = to_t_points(open_obc(path)).isel(time=slice(0, NTIME))
    SEG_RAW[seg.attrs["segment"]] = seg

seg_summary = pd.DataFrame([dict(
    segment=sid,
    orientation=ds.attrs["orientation"],
    const_coord=round(ds.attrs["const_coord"], 4),
    n_points=ds.sizes["n"],
    n_levels=ds.sizes["depth"],
    n_times=ds.sizes["time"],
    obc_max_depth=round(float(ds["depth"].max()), 1),
    start=str(ds["time"].values[0])[:10],
    end=str(ds["time"].values[-1])[:10],
    label=obc_label(ds),
) for sid, ds in SEG_RAW.items()]).set_index("segment")

print(f"{CASE['casename']}: {len(SEG_RAW)} segment(s), first {NTIME} time record(s) loaded")
seg_summary


## 2. The sea floor, and masking what lies below it

Now that each segment sits on the model's tracer cells, the row (or column) of `<inputdir>/ocean_topog_*.nc` along the boundary lines up with it point for point -- `on_boundary` just slices it out. Blank everything below the local sea floor (comparing each level *centre* depth with the floor depth) and every point on land.


In [ ]:
# from mom6_tools.regional_m6toolbox import on_boundary, mask_below_floor

FLOOR = {sid: on_boundary(TOPOG, ds, TOPOG) for sid, ds in SEG_RAW.items()}
SEG = {sid: mask_below_floor(ds, FLOOR[sid]) for sid, ds in SEG_RAW.items()}
FLOOR_MAX = {sid: float(f["depth"].max()) for sid, f in FLOOR.items()}

mask_summary = pd.DataFrame([dict(
    segment=sid,
    n_points=SEG_RAW[sid].sizes["n"],
    n_wet=int((FLOOR[sid]["mask"] > 0).sum()),
    n_land=int((FLOOR[sid]["mask"] <= 0).sum()),
    floor_max=round(FLOOR_MAX[sid], 1),
    floor_mean_wet=round(float(FLOOR[sid]["depth"].where(FLOOR[sid]["mask"] > 0).mean()), 1),
    cells_total=int(SEG_RAW[sid]["temp"].size),
    kept_frac=round(float(SEG[sid]["temp"].notnull().mean()), 4),
) for sid in SEG_RAW]).set_index("segment")
mask_summary


In [ ]:
# from mom6_tools.regional_m6toolbox import along_name, obc_label

fig, axs = plt.subplots(len(SEG), 1, figsize=(12, 2.8 * len(SEG)), squeeze=False)
for ax, (sid, ds) in zip(axs.ravel(), SEG_RAW.items()):
    floor = FLOOR[sid]
    xn = along_name(ds)
    land = floor["mask"] <= 0

    ax.fill_between(floor[xn].values, floor["depth"].values, 0.0, color="0.85")
    floor["depth"].plot(ax=ax, x=xn, color="k", lw=1.0, label="sea-floor depth")
    ax.plot(floor[xn].where(land).values, np.zeros(floor.sizes["n"]), "|",
            color="firebrick", ms=10,
            label=f"land ({int(land.sum())} of {floor.sizes['n']} points)")
    ax.invert_yaxis()
    ax.set_ylabel("depth [m]")
    ax.set_xlabel(f"{xn} [deg]")
    ax.set_title(f"{obc_label(ds)} — sea floor along the boundary")
    ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIGDIR / "02_segment_bathymetry.png", dpi=130)
plt.show()


Everything below uses the masked segments in `SEG`; the unmasked originals stay in `SEG_RAW` for comparison.

## 3. Where the segments sit

Model bathymetry with each segment's `lon`/`lat` drawn on top.

In [ ]:
# from mom6_tools.regional_m6toolbox import obc_label

fig, ax = plt.subplots(figsize=PANEL, subplot_kw=SUBPLOT)

STATIC["deptho"].where(STATIC.wet).plot(ax=ax, **MAP["h"], cmap="cmo.deep",
            cbar_kwargs={"label": "deptho [m]"})
ax.coastlines("50m")
ax.add_feature(LAND, facecolor="0.85")
gl = ax.gridlines(**GRID)

colors = plt.cm.tab10.colors
for i, (sid, ds) in enumerate(SEG.items()):
    c = colors[i % 10]
    lon, lat = ds["lon"].values, ds["lat"].values
    ax.plot(lon, lat, color=c, lw=2.5, transform=ccrs.PlateCarree(),
            label=obc_label(ds), zorder=5)
    mid = len(lon) // 2
    ax.text(float(lon[mid]), float(lat[mid]), f" {sid}", color=c, fontsize=11,
            fontweight="bold", transform=ccrs.PlateCarree(), zorder=6,
            ha="center", va="bottom")
ax.legend(loc="lower left", fontsize=8, framealpha=0.9)

# ax.set_title does not render on a Cartopy axis with labelled gridlines
fig.suptitle(f"{CASE['casename']}: bathymetry and open boundary segments")
fig.savefig(FIGDIR / "02_segment_map.png", dpi=130)
plt.show()

## 4. Sections along each boundary

Depth against along-boundary position for temperature, salinity and the normal velocity at one time, plus the 1-D sea-surface height. The black line is the sea floor; the axis is clipped to the deepest point on the segment.


In [ ]:
# from mom6_tools.regional_m6toolbox import along_name, normal_var, obc_label

SEC_TIME = 0          # index into the loaded OBC time records

for sid, ds in SEG.items():
    xn = along_name(ds)
    nv = normal_var(ds)

    fig, axs = plt.subplots(2, 2, figsize=(13, 7.5), sharex=True)
    panels = [("temp", dict(cmap="cmo.thermal")),
              ("salt", dict(cmap="cmo.haline")),
              (nv, DIFF)]

    for ax, (v, kw) in zip(axs.ravel(), panels):
        ds[v].isel(time=SEC_TIME).plot(ax=ax, x=xn, y="depth", **SECTION, **kw)
        FLOOR[sid]["depth"].plot(ax=ax, x=xn, color="k", lw=1.2)
        ax.set_ylim(1.05 * FLOOR_MAX[sid], 0.0)
        ax.set_ylabel("depth [m]")
        ax.set_title(v + ("  (normal to the boundary)" if v == nv else ""))

    axe = axs.ravel()[3]
    eta = ds["eta"]
    axe.fill_between(eta[xn].values, eta.min("time").values, eta.max("time").values,
                     color="0.8", label=f"range over {ds.sizes['time']} records")
    eta.isel(time=SEC_TIME).plot(ax=axe, x=xn, lw=1.0, color="C0", label="eta")
    axe.axhline(0.0, color="0.5", lw=0.6)
    axe.set_ylabel("eta [m]")
    axe.set_title("eta")
    axe.legend(fontsize=8)

    for ax in axs.ravel():
        ax.set_xlabel(f"{xn} [deg]")
    fig.suptitle(f"{obc_label(ds)} — {str(ds['time'].values[SEC_TIME])[:10]}")
    fig.tight_layout()
    fig.savefig(FIGDIR / f"02_section_{sid}.png", dpi=130)
    plt.show()


## 5. Time series along each boundary

The mean and the maximum of each forcing field over the loaded records, reduced over every cell that survived the sea-floor mask: the along-boundary points `n` and, for the 3-D fields, the levels above the local floor. A mean that drifts, or a maximum that walks away from the rest of the record, is what a bad interpolation or a stray fill value looks like from here.

In [ ]:
# from mom6_tools.regional_m6toolbox import normal_var, obc_label

# --- Mean, min and max of each boundary field over the loaded time records ---
# Reduced over every unmasked cell of the segment: the along-boundary points `n`
# and, for the 3-D fields, the levels `depth` that survived the sea-floor mask.
# The OBC files carry no `units` attribute, so label the axes here.
UNITS = {"temp": "degC", "salt": "psu", "u": "m s-1", "v": "m s-1", "eta": "m"}

for sid, ds in SEG.items():
    nv = normal_var(ds)

    fig, axs = plt.subplots(2, 2, figsize=(13, 6.5), sharex=True)
    for ax, v in zip(axs.ravel(), ["temp", "salt", nv, "eta"]):
        da = ds[v]
        dims = [d for d in ("n", "depth") if d in da.dims]    # eta is 2-D, so `n` only
        da.mean(dims).plot(ax=ax, x="time", color="black", lw=1.2,
                           label="mean over " + "+".join(dims))
        da.max(dims).plot(ax=ax, x="time", color="red", lw=1.2, ls="--", label="max")
        da.min(dims).plot(ax=ax, x="time", color="blue", lw=1.2, ls="--", label="min")
        if v in (nv, "eta"):
            ax.axhline(0.0, color="0.5", lw=0.6)
        ax.set_ylabel(f"{v} [{UNITS[v]}]")
        ax.set_title(v + ("  (normal to the boundary)" if v == nv else ""))
        ax.set_xlabel("")
        ax.legend(fontsize=8)

    for ax in axs[-1]:
        ax.set_xlabel("time")
        ax.tick_params(axis="x", labelrotation=30)
    fig.suptitle(f"{obc_label(ds)} — mean, min and max over {ds.sizes['time']} records")
    fig.tight_layout()
    fig.savefig(FIGDIR / f"02_timeseries_{sid}.png", dpi=130)
    plt.show()


## 6. The boundary against the interior

Side-by-side Hovmöller diagrams for one variable — time across, along-boundary position up: the OBC file on the left, the row (or column) of interior h-points along the same boundary in the middle, and their difference on the right. `on_boundary` picks out that interior line, and because the segment is already on the tracer cells the two sides share the same `n` and subtract directly.

The OBC records are daily, so the comparison uses the daily `sfc` stream at the surface; `SSU`/`SSV` are moved onto the tracer point first, so a single interior line serves every variable.


In [ ]:
# from mom6_tools.regional_m6toolbox import decode_mom6_time, make_grid

# --- The surface history stream, everything moved onto the tracer point ---
# The OBC segments are daily, so the daily `sfc` stream is the one that lines up
# with them.  SSU/SSV are interpolated off their cell faces so that a single
# interior row (or column) of h points can be sliced for every variable.
sfc = xr.open_mfdataset(f"{FILE_PATTERN}.h.sfc.*.nc", decode_times=False,
                        chunks={"time": 1})
sfc = decode_mom6_time(sfc, calendar=MOM6_CALENDAR).sel(time=slice(*DATE_RANGE))
grid, sfc_metrics = make_grid(sfc, GEOM)

SSH_NAME = "SSH" if "SSH" in sfc else "zos"     # carib12 writes SSH, small_alaska zos
SFC_H = xr.Dataset({
    "temp": sfc["tos"],
    "salt": sfc["sos"],
    "eta":  sfc[SSH_NAME],
    "u": grid.interp(sfc_metrics["SSU"].where(STATIC.wet_u), "X").reset_coords(drop=True),
    "v": grid.interp(sfc_metrics["SSV"].where(STATIC.wet_v), "Y").reset_coords(drop=True),
}).where(STATIC.wet)

print(f"sfc: {SFC_H.sizes['time']} records, "
      f"{str(SFC_H.time.values[0])[:10]} -> {str(SFC_H.time.values[-1])[:10]}, "
      f"sea surface height from {SSH_NAME!r}")

In [ ]:
# from mom6_tools.regional_m6toolbox import on_boundary, along_name, normal_var, obc_label

HOV_VAR = "temp"    # "temp", "salt", "eta", or "normal" for the boundary-normal velocity

CMAP = {"temp": "cmo.thermal", "salt": "cmo.haline"}

for sid, obc in SEG.items():
    v = normal_var(obc) if HOV_VAR == "normal" else HOV_VAR
    xn = along_name(obc)

    # The OBC surface level, and the interior h-points along the same boundary.
    ob = obc[v].isel(depth=0) if "depth" in obc[v].dims else obc[v]
    line = on_boundary(SFC_H[v], obc, TOPOG).load()

    # OBC records are snapshots at 00:00, history records daily means stamped
    # 12:00: clip to the window they share, then put the OBC on the history times.
    t0 = max(ob["time"].values[0], line["time"].values[0])
    t1 = min(ob["time"].values[-1], line["time"].values[-1])
    ob, line = ob.sel(time=slice(t0, t1)), line.sel(time=slice(t0, t1))
    diff = ob.interp(time=line["time"]) - line

    # Shared, outlier-tolerant colour range for the two data panels.
    lo = min(float(ob.quantile(0.02)), float(line.quantile(0.02)))
    hi = max(float(ob.quantile(0.98)), float(line.quantile(0.98)))
    if v in ("u", "v", "eta"):
        lim = max(abs(lo), abs(hi))
        kw = dict(cmap="cmo.balance", vmin=-lim, vmax=lim)
    else:
        kw = dict(cmap=CMAP[v], vmin=lo, vmax=hi)

    fig, axs = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
    for ax, (da, title, pkw) in zip(axs, [
            (ob,   f"OBC {v}" + ("  (surface level)" if "depth" in obc[v].dims else ""), kw),
            (line, f"interior {v} ({line.attrs['boundary_line']})", kw),
            (diff, "OBC - interior", DIFF)]):
        da.plot(ax=ax, x="time", y=xn, **pkw,
                cbar_kwargs={"label": f"{v} [{UNITS[v]}]", "orientation": "horizontal",
                             "pad": 0.18})
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(axis="x", labelrotation=30)
    axs[0].set_ylabel(f"{xn} [deg]")
    fig.suptitle(f"{obc_label(obc)} — {v} along the boundary vs. the interior")
    fig.tight_layout()
    fig.savefig(FIGDIR / f"02_hovmoller_{v}_{sid}.png", dpi=130)
    plt.show()


## Cleanup

In [ ]:
done = False # let's you use Run All without closing the cluster.
if done:
    client.close()
    cluster.close()